# 02. Data Engineering: Synthetic Sudoku Digit Dataset Generation

### Overview
High accuracy digit recognition on real-world Sudoku puzzles requires training data that reflects physical print characteristics:
- Diverse typography (serif, sans-serif, monospaced fonts)
- Paper grain, Gaussian noise, and blur
- Uneven lighting and contrast variations
- Edge fragments from grid borders

This notebook synthesizes thousands of realistic $28 \times 28$ grayscale images across classes `1` through `9` and `empty`.

In [ ]:
import os
import sys
import random
from pathlib import Path

def find_root() -> Path:
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / "models" / "digit_classifier.keras").exists():
            return parent
    return Path(os.path.abspath("..")).resolve()

PROJECT_ROOT = find_root()
DIGITS_DIR = PROJECT_ROOT / "data" / "digits"

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

print("Target Dataset Directory:", DIGITS_DIR)

## 1. Augmentation & Realism Engine
The `make_realistic` function transforms synthetic canvas renders into realistic scanned camera crops by introducing brightness variance, partial grid lines, blur, and random pixel noise.

In [ ]:
def make_realistic(img_pil: Image.Image) -> np.ndarray:
    """Applies photometric variations, noise, and edge artifacts to a PIL image."""
    arr = np.array(img_pil, dtype=np.float32)
    
    # 1. Random contrast and brightness
    arr = arr * random.uniform(0.65, 1.0) + random.uniform(0, 35)
    
    # 2. Gaussian blur
    if random.random() < 0.5:
        k = random.choice([3, 5])
        arr = cv2.GaussianBlur(arr, (k, k), 0)
        
    # 3. Grid line artifacts on 1-2 borders
    for _ in range(random.randint(0, 2)):
        t = random.randint(1, 2)
        side = random.choice(['top', 'bottom', 'left', 'right'])
        v = random.randint(0, 80)
        if side == 'top':    arr[:t, :] = v
        elif side == 'bottom': arr[-t:, :] = v
        elif side == 'left':   arr[:, :t] = v
        elif side == 'right':  arr[:, -t:] = v
        
    # 4. Salt-and-pepper / Gaussian noise
    noise = np.random.normal(0, random.uniform(2, 8), arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    
    return arr

## 2. Empty Cell Generation
Generating realistic empty cell crops (paper background with occasional faint lines or specks).

In [ ]:
def generate_empty_cells(count: int = 500):
    empty_dir = DIGITS_DIR / "empty"
    empty_dir.mkdir(parents=True, exist_ok=True)
    
    for i in range(count):
        base_color = random.randint(220, 255)
        img = Image.new("L", (28, 28), color=base_color)
        realistic = make_realistic(img)
        cv2.imwrite(str(empty_dir / f"empty_{i:04d}.png"), realistic)
        
    print(f"Generated {count} empty cell images in {empty_dir}")

generate_empty_cells(count=500)

## 3. Digit Synthesis (Classes 1–9)
We render digits `1` through `9` using available system fonts, random slight rotations, scaling, and the realism pipeline.

In [ ]:
# Discover available system fonts
FONT_CANDIDATES = [
    "arial.ttf", "calibri.ttf", "tahoma.ttf", "times.ttf",
    "consola.ttf", "segoeui.ttf", "DejaVuSans.ttf"
]
AVAILABLE_FONTS = []
for font_name in FONT_CANDIDATES:
    try:
        f = ImageFont.truetype(font_name, 22)
        AVAILABLE_FONTS.append(font_name)
    except IOError:
        continue

if not AVAILABLE_FONTS:
    print("Warning: No candidate TrueType fonts found. Using default PIL font.")

def generate_digit_images(samples_per_digit: int = 500):
    for digit in range(1, 10):
        digit_dir = DIGITS_DIR / str(digit)
        digit_dir.mkdir(parents=True, exist_ok=True)
        
        for i in range(samples_per_digit):
            # Render on 28x28 white background
            bg_color = random.randint(225, 255)
            img = Image.new("L", (28, 28), color=bg_color)
            draw = ImageDraw.Draw(img)
            
            # Select font and size
            font_size = random.randint(18, 22)
            font = None
            if AVAILABLE_FONTS:
                font = ImageFont.truetype(random.choice(AVAILABLE_FONTS), font_size)
            
            # Center digit with slight jitter
            offset_x = random.randint(-2, 2)
            offset_y = random.randint(-2, 2)
            digit_color = random.randint(0, 45)
            
            draw.text((7 + offset_x, 2 + offset_y), str(digit), fill=digit_color, font=font)
            
            # Small random rotation (-10 to +10 deg)
            angle = random.uniform(-10, 10)
            img = img.rotate(angle, fillcolor=bg_color)
            
            realistic = make_realistic(img)
            cv2.imwrite(str(digit_dir / f"digit_{digit}_{i:04d}.png"), realistic)
            
    print(f"Generated {samples_per_digit * 9} digit images across classes 1-9.")

generate_digit_images(samples_per_digit=500)

## 4. Dataset Verification & Sample Inspection
Examine class distribution counts and preview representative augmented patches.

In [ ]:
classes = sorted([d.name for d in DIGITS_DIR.iterdir() if d.is_dir()])
print(f"Found {len(classes)} classes: {classes}\n")

preview_samples = []
for cls in classes:
    files = list((DIGITS_DIR / cls).glob("*.png"))
    print(f"  Class '{cls}': {len(files)} images")
    if files:
        preview_samples.append((cls, cv2.imread(str(files[0]), cv2.IMREAD_GRAYSCALE)))

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, (cls, img_patch) in zip(axes.flat, preview_samples):
    ax.imshow(img_patch, cmap="gray")
    ax.set_title(f"Class: {cls}")
    ax.axis("off")
plt.suptitle("Synthesized Dataset Samples", fontsize=12)
plt.tight_layout()
plt.show()